## データセットダウンロード(一回だけ実施)

pip install gdwon


In [ ]:
import os
import gdown
import tarfile

# 1. ダウンロード対象のファイルIDと出力先
file_id = "1GwClT3GG2y190ydN7eFObrl1jubUmykd"
url = f"https://drive.google.com/uc?id={file_id}"
output_path = "./downloaded_folder/archive.tar"

# 2. 出力先フォルダ作成
os.makedirs("./downloaded_folder", exist_ok=True)

# 3. gdownでダウンロード
gdown.download(url, output_path, quiet=False)

# 4. tarファイルを解凍（拡張子が .tar の場合）
with tarfile.open(output_path, "r") as tar:
    tar.extractall(path="./downloaded_folder")

print("ダウンロードと展開が完了しました。")


In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import librosa
import soundfile as sf

#dataを作成するプログラム群
output_folder = "./../dataset"

#ファイル階層は以下を想定
#
# behavior(data_folder)
#    |
#    |--text_ful
#    |     |--MYN01_ful.txt
#    |      --MCN01_ful.txt
#     --  wav_ful
#          |--MYN01_ful.wavs
#           --MCN01_ful.wav


# 展開したデータフォルダ
old_data_folder = "./downloaded_folder/Dataset_enge_ラベル&データ確認済"


#ファイル階層は以下を想定
#
# EAT_old(data_folder)
#    |
#    |--A
#    |  |--001_CBG_22k_MAN01_1.wav
#    |   --001_CBG_MAN01_01.txt
#     --B
#       |--001_CBG_22k_MHT01_1.wav
#        --001_CBG_MHT01_01.txt

wav_output_folder = old_data_folder + "/wav_div"
text_output_folder = old_data_folder + "/text_strong_div"


# 利用したいマイクのリスト
Mics = {
    #"SM": 1,
    # "EM01": 2,
    # "EM02": 3,
    # "TM01": 4,
    "TM02": 5,
    # "CM": 6
}

Augmentation = [
    #"ngram", 未実装
    #"label_start_cut",
    #"enhancement" #4ch-summed signal
]

#切り分ける音声ファイルの長さと最大値を設定
segment = 10
max_segment_sec = 10


old_wav_folder = output_folder + "/old_wav_ful"
old_txt_folder = output_folder + "/old_txt_ful"
old_div_folder = output_folder + "/old_wav_aug"
old_div_text_folder = output_folder + "/old_text_aug"



## ***データを取り出し、各マイクフォルダへと割り当てる

In [2]:



# 各フォルダに対して処理を行う
for folder in Mics:
    # 完全なパスを作成
    folder_path = os.path.join(old_wav_folder, folder)
    
    # ディレクトリが存在しない場合は作成
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        #print(f"Created folder: {folder_path}")
    else:
        print(f"Folder already exists: {folder_path}")

if not os.path.exists(old_txt_folder):
    os.makedirs(old_txt_folder)
    #print(f"Created folder: {old_txt_folder}")
else:
    print(f"Folder already exists: {old_txt_folder}")

import shutil

def move_text_files(source_folder, destination_folder):
    # source_folder の中を再帰的に探索
    for dirpath, dirnames, filenames in os.walk(source_folder):
        for filename in filenames:
            if not filename.startswith('._') and filename.endswith('.txt'):  # テキストファイルをチェック
                source_file = os.path.join(dirpath, filename)
                filename = filename.replace("001_", "")
                destination_file = os.path.join(destination_folder, filename)


                # ファイルを移動する前に、同名のファイルが目的地に存在するか確認
                if os.path.exists(destination_file):
                    print(f"Error: {destination_file} already exists.")
                else:
                    shutil.copy(source_file, destination_file)  # ファイルを移動
                    #print(f"copy: {source_file} -> {destination_file}")

def move_wav_files(source_folder, destination_folder):
    # source_folder の中を再帰的に探索
    for dirpath, dirnames, filenames in os.walk(source_folder):
        print(dirpath)
        for filename in tqdm(filenames):
            if not filename.startswith('._') and filename.endswith('.wav'):  # テキストファイルをチェック
                for mic in Mics:
                    source_file = os.path.join(dirpath, filename)
                    #filename = filename.replace("001_", "")
                    mic_num = source_file.split("_")[-1].split(".")[0]
                    if str(Mics[mic]) == mic_num:
                        filename = mic + "/" +filename
                        destination_file = os.path.join(destination_folder, filename)
                        #print(destination_file)

                        # ファイルを移動する前に、同名のファイルが目的地に存在するか確認
                        if os.path.exists(destination_file):
                            print(f"Error: {destination_file} already exists.")
                        else:
                            shutil.copy(source_file, destination_file)  # ファイルを移動
                            #print(f"copy: {source_file} -> {destination_file}")


def resample_audio_files(source_folder, target_folder, target_sr=16000):
    # source_folder の中を再帰的に探索
    for dirpath, dirnames, filenames in os.walk(source_folder):
        print(dirpath)
        for filename in tqdm(filenames):
            if filename.lower().endswith(('.wav', '.flac', '.mp3')):  # 対応する音声ファイル形式
                source_file = os.path.join(dirpath, filename)


                filename = filename.replace("_22k_", "_")
                filename = filename.replace("001_", "")
                target_file = os.path.join(dirpath, filename)

                
                # 音声を読み込み、サンプリングレートを変更
                audio, sr = librosa.load(source_file, sr=None)  # 元のサンプリングレートで読み込み
                if sr != target_sr:
                    audio_resampled = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
                    # レート変更後の音声を保存
                    sf.write(target_file, audio_resampled, target_sr)
                    #print(f"Resampled and saved {source_file} -> {target_file} at {target_sr} Hz")
                    if os.path.exists(source_file):
                        # ファイルを削除
                        os.remove(source_file)
                        #print(f"Deleted: {source_file}")
        
                else:
                    # サンプリングレートが既に目的のものであれば、何もしない
                    #print(f"nothing to do {source_file} -> {target_file}")
                    continue
                
source = old_data_folder
destination = old_txt_folder
print("move text files!")
move_text_files(source, destination)
print("finish text files!")

source = old_data_folder
destination = old_wav_folder
print("move wav files!")
move_wav_files(source, destination)
print("finish wav files!")

source = old_wav_folder
destination = old_wav_folder
print("resample audio_files!")
resample_audio_files(source, destination)
print("all finished!")

move text files!
finish text files!
move wav files!
./downloaded_folder/Dataset_enge_ラベル&データ確認済


100%|██████████| 1/1 [00:00<00:00, 46603.38it/s]


./downloaded_folder/Dataset_enge_ラベル&データ確認済/F


100%|██████████| 191/191 [00:00<00:00, 1663.80it/s]


./downloaded_folder/Dataset_enge_ラベル&データ確認済/E


...

finish wav files!
resample audio_files!
./../dataset/old_wav_ful


0it [00:00, ?it/s]


./../dataset/old_wav_ful/TM02


100%|██████████| 148/148 [00:07<00:00, 20.49it/s]

all finished!


# **データフォルダからテキスト情報を収集、DFに格納**

In [3]:
import os
import pandas as pd
import re

text_folder = old_txt_folder

# フォルダ内の全てのテキストファイルのリストを取得
files = [f for f in os.listdir(text_folder) if f.endswith('.txt')]

# 全ファイルのデータを保持するリスト
all_rows = []

# 各ファイルに対して処理を行う
for file_name in files:
    file_path = os.path.join(text_folder, file_name)
    speaker = file_name.split("_")[-2]
    food = file_name.split("_")[0]
    
    if len(file_name.split("_")) == 4:
        index = file_name.split("_")[1]
        food = food + "_" + index
    
    # ファイルを開いて各行をリストとして読み込む
    with open(file_path, 'r', encoding='utf-8') as file:
         rows = [
                    [float(row[0])] + [float(row[1])] + row[2:]
                    for row in (
                        [item for item in re.split(r'\s+', line.strip()) if item]  # 連続する空白で分割し、空の要素を除外
                        for line in file
                    )
                ]
    
    # 読み込んだ各行に話者名を追加し、全体のリストに追加
    for row in rows:
        all_rows.append([speaker] + [food] + row)

# 全データを含むデータフレームを作成
df = pd.DataFrame(all_rows, columns=['Speaker', "Food", 'StartTime','EndTime', 'Behavior']) 

# データフレームを表示

print(f"総行数: {len(df)}")
print(df)  # データフレームの最初の数行を表示

総行数: 24193
      Speaker   Food  StartTime    EndTime Behavior
0       MYY01  W20_2   5.767372   6.285392  swallow
1       MYY01  W20_2  13.743098  14.182735  swallow
2       MYY01  W20_2  21.310951  21.887815  swallow
3       MYY01  W20_2  28.737482  29.244744  swallow
4       MYY01  W20_2  36.497481  36.943934  swallow
...       ...    ...        ...        ...      ...
24188   MKG01  W20_2  31.389235  32.111970  swallow
24189   MKG01  W20_2  39.394583  39.918701  swallow
24190   MKG01  W20_2  53.036869  53.649158  swallow
24191   MKG01  W20_2  61.395967  61.939097  swallow
24192   MKG01  W20_2  69.777879  70.204555  swallow

[24193 rows x 5 columns]


In [4]:
#複数回実行するとエラーを吐くので最初から実行し直して！！

speaker_list = df['Speaker'].unique().tolist()
rows = []

for i in range(len(speaker_list)):

    foods = df[df.Speaker== speaker_list[i]].Food.unique().tolist()
    for food in foods:
        start = df[(df.Speaker == speaker_list[i]) & (df.Food == food)].StartTime.min()
        end = df[(df.Speaker==speaker_list[i]) & (df.Food == food)].EndTime.max()
        
        right_len = len(df[(df.Speaker==speaker_list[i]) & (df.Behavior=="right") & (df.Food == food)])
        left_len = len(df[(df.Speaker==speaker_list[i]) & (df.Behavior=="left") & (df.Food == food)])
        chew_len = right_len + left_len
        swallow_len = len(df[(df.Speaker==speaker_list[i]) & (df.Behavior=="swallow") & (df.Food == food)])
        #all_data_len = len(df[df.Speaker==speaker_list[i]])
    
        rows.append([speaker_list[i], food, left_len, right_len, chew_len, swallow_len, start, end])

df_count = pd.DataFrame(rows, columns=['Speaker', "Food", 'Left', 'Right', 'Chewing', 'Swallowing', 'StartTime', 'EndTime']) 
print(df_count)

    Speaker   Food  Left  Right  Chewing  Swallowing  StartTime     EndTime
0     MYY01  W20_2     0      0        0          10   5.767372   79.167667
1     MYY01    CBG   381    176      557          23   1.213323  423.436198
2     MYY01  RTZ_2   253    152      405          12   0.912516  324.390571
3     MYY01  W20_1     0      0        0          10   5.110104   82.751744
4     MYY01    GUM   101    117      218           6   5.764389  181.058089
..      ...    ...   ...    ...      ...         ...        ...         ...
143   MRS01  W20_1     0      0        0          10   2.093939   62.975908
144   MTN02  W20_1     0      0        0           9   4.917244   66.639073
145   MTN02  RTZ_1   151    207      358          14   1.870696  316.814578
146   MTN02  RTZ_2   233     34      267          11   0.789955  245.870308
147   MTN02    GUM   189     55      244          10   2.397778  171.375331

[148 rows x 8 columns]


In [5]:
import os

if "enhancement" in Augmentation:
    folder = "conbined"
    folder_path = os.path.join(old_div_folder, folder)
    
    # ディレクトリが存在しない場合は作成
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"Created folder: {folder_path}")
    else:
        print(f"Folder already exists: {folder_path}")

else:
    # 各フォルダに対して処理を行う
    for folder in Mics:
        # 完全なパスを作成
        folder_path = os.path.join(old_div_folder, folder)
        
        # ディレクトリが存在しない場合は作成
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)
            print(f"Created folder: {folder_path}")
        else:
            print(f"Folder already exists: {folder_path}")

if not os.path.exists(old_div_text_folder):
        os.makedirs(old_div_text_folder)
        print(f"Created folder: {old_div_text_folder}")
else:
    print(f"Folder already exists: {old_div_text_folder}")


Created folder: ./../dataset/old_wav_aug/TM02
Folder already exists: ./../dataset/old_text_aug


In [6]:
import os
import numpy as np
import librosa
import soundfile as sf
from tqdm import tqdm

# ======== 設定（ここだけ書き換えればOK） ========
fr = 16000               # サンプリングレート
window_length = 10.0     # 切り出すウィンドウ長（秒）
overlap = 2.0           # 重複長（秒）
step = window_length - overlap
# ==============================================

# 既存の変数: df, Mics, old_wav_folder, old_div_folder, old_div_text_folder, Augmentation, folder

speaker_list = df['Speaker'].unique().tolist()

for Speaker in tqdm(speaker_list, desc='Speakers'):
    foods = df[df.Speaker == Speaker].Food.unique().tolist()
    for Food in foods:
        sp_df = df[(df.Speaker == Speaker) & (df.Food == Food)].reset_index(drop=True)
        # 各マイクファイルのパスリスト
        mic_paths = [f"{old_wav_folder}/{mic}/{Food}_{Speaker}_{Mics[mic]}.wav" for mic in Mics]
        # 総再生時間を取得（最初のマイクファイルを利用）
        total_duration = librosa.get_duration(filename=mic_paths[0], sr=fr)

        start = 0.0
        while start + window_length <= total_duration:
            end = start + window_length
            # セグメント開始時刻を文字列化
            short_start = round(start, 2)
            integer = str(short_start).split('.')[0].rjust(4, '0')
            decimal = str(short_start).split('.')[1].ljust(2, '0')

            # オーディオ読み込み・合成
            if 'enhancement' in Augmentation:
                y_combined, sr = librosa.load(mic_paths[0], sr=fr, offset=start, duration=window_length)
                for mp in mic_paths[1:]:
                    y, _ = librosa.load(mp, sr=fr, offset=start, duration=window_length)
                    y_combined += y
                segment_audio = y_combined
                # 保存
                wav_file = f"{old_div_folder}/{folder}/eat_{Speaker}_{folder}_{Food}_{integer}_{decimal}.wav"
                sf.write(wav_file, segment_audio, sr)
            else:
                for mic in Mics:
                    mp = f"{old_wav_folder}/{mic}/{Food}_{Speaker}_{Mics[mic]}.wav"
                    y, sr = librosa.load(mp, sr=fr, offset=start, duration=window_length)
                    segment_audio = y
                    wav_file = f"{old_div_folder}/{folder}/eat_{Speaker}_{folder}_{Food}_{integer}_{decimal}.wav"
                    sf.write(wav_file, segment_audio, sr)

            # ラベルファイル作成
            txt_file = f"{old_div_text_folder}/eat_{Speaker}_{Food}_{integer}_{decimal}.txt"
            with open(txt_file, 'w') as out:
                for _, ev in sp_df.iterrows():
                    ev_s, ev_e, label = ev.StartTime, ev.EndTime, ev.Behavior
                    # セグメントとイベントが重なる場合のみ
                    if ev_e > start and ev_s < end:
                        rel_s = max(ev_s, start) - start
                        rel_e = min(ev_e, end) - start
                        tag = {
                            'right': 'ch', 'left': 'ch', 'front': 'ch',
                            'swallow': 'sw', 'noise': 'no', 'other': 'no'
                        }.get(label)
                        if tag:
                            out.write(f"{rel_s}\t{rel_e}\t{tag}\n")
            # 次のウィンドウへ移動
            start += step


Speakers:   0%|          | 0/32 [00:00<?, ?it/s]/tmp/ipykernel_2190749/472995089.py:25: FutureWarning: get_duration() keyword argument 'filename' has been renamed to 'path' in version 0.10.0.
	This alias will be removed in version 1.0.
  total_duration = librosa.get_duration(filename=mic_paths[0], sr=fr)
Speakers: 100%|██████████| 32/32 [00:24<00:00,  1.29it/s]


In [7]:
import os
import wave

def list_wav_longer_than_20s(folder_path):
    long_durations = []
    long_files_full_path = []  # フルパスを格納する配列

    # フォルダ内のすべてのファイルを再帰的にリストアップ
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            # ファイルがWAV形式であるか確認
            if file.endswith('.wav'):
                # WAVファイルのフルパスを取得
                file_path = os.path.join(root, file)
                
                # WAVファイルを開く
                try:
                    with wave.open(file_path, 'r') as wav_file:
                        # フレーム数を取得
                        frames = wav_file.getnframes()
                        # レート（1秒あたりのフレーム数）を取得
                        rate = wav_file.getframerate()
                        # 持続時間を計算
                        duration = frames / float(rate)
                        
                        # 持続時間が20秒以上の場合にリストに追加
                        if duration > 17:
                            long_durations.append((file_path, duration))  # ファイルのフルパスを保存
                except wave.Error as e:
                    print(f"Error opening {file_path}: {e}")

    # 持続時間でリストを降順に並び替え
    long_durations.sort(key=lambda x: x[1], reverse=True)
    
    # 並び替えた結果を出力し、フルパスを配列に格納
    for file_path, duration in long_durations:
        long_files_full_path.append(file_path)
        print(f"{file_path}: {duration}秒")

    return long_files_full_path

# 使用例
folder_path = wav_output_folder
long_files = list_wav_longer_than_20s(folder_path)
"""
# 出力されたファイルパスを使ってファイルを削除
for file_path in long_files:
    print(f"Deleting {file_path}...")
    os.remove(file_path)
    print(f"Deleted {file_path}")
    """

print("All specified files have been deleted.")


All specified files have been deleted.


In [8]:
long_files

[]

In [9]:
import os
import shutil
from pathlib import Path
import numpy as np
from scipy import signal
from scipy.io.wavfile import read, write

# 設定
folder_path = old_div_folder  # 検索するフォルダのパス
cutoff = 100.0  # カットオフ周波数
numtaps = 1023  # FIRフィルタの長さ（奇数が推奨）

def high_pass_filter(input_path: Path, output_path: Path, cutoff: float, numtaps: int) -> None:
    """入力音声に対してHigh-Pass Filter（HPF）を適用し、結果を別のファイルに保存する。

    Args:
        input_path (Path): 入力音声のパス。
        output_path (Path): 出力音声の保存先パス。
        cutoff (float): カットオフ周波数。
        numtaps (int): FIRフィルタのタップ数（フィルタの長さ）。
    """
    fs, data = read(str(input_path))
    # データを-1.0から1.0の範囲に正規化
    data = data.astype(np.float32)
    max_int16 = 2**15
    data /= max_int16

    # 高域通過フィルタの設計
    hpf = signal.firwin(numtaps, cutoff, pass_zero=False, fs=fs)
    
    # フィルタを適用
    filtered_data = signal.lfilter(hpf, 1.0, data)
    
    # データをint16に戻す
    filtered_data = np.clip(filtered_data, -1, 1)
    filtered_data = (filtered_data * max_int16).astype(np.int16)
    
    write(str(output_path), fs, filtered_data)

def apply_filter_to_files(folder_path: str, cutoff: float, numtaps: int) -> None:
    """指定したフォルダ内のすべてのWAVファイルにHPFを適用し、元のファイルを上書きする。

    Args:
        folder_path (str): WAVファイルが存在するフォルダのパス。
        cutoff (float): カットオフ周波数。
        numtaps (int): フィルタのタップ数。
    """
    for root, dirs, files in os.walk(folder_path):
        for file in tqdm(files):
            if file.endswith(".wav"):
                full_path = Path(root) / file
                temp_path = Path(root) / f"temp_{file}"
                #print(f"Processing: {full_path}")
                
                # フィルタリングを実施
                high_pass_filter(full_path, temp_path, cutoff, numtaps)
                
                # 元のファイルを削除して、新しいファイル名を元のファイル名に変更
                os.remove(full_path)
                shutil.move(temp_path, full_path)
                #print(f"Updated: {full_path}")

# 実行
apply_filter_to_files(folder_path, cutoff, numtaps)


0it [00:00, ?it/s]
  0%|          | 0/3142 [00:00<?, ?it/s]

100%|██████████| 3142/3142 [00:33<00:00, 92.57it/s]


## 余分なフォルダの削除

In [10]:


def delete_folder_recursively(folder_path):
    # フォルダが存在するかを確認
    if os.path.exists(folder_path):
        # フォルダを再帰的に削除
        shutil.rmtree(folder_path)
        print(f"フォルダ '{folder_path}' を再帰的に削除しました。")
    else:
        print(f"指定されたフォルダ '{folder_path}' は存在しません。")

delete_folder_recursively(old_wav_folder)
delete_folder_recursively(old_txt_folder)


フォルダ './../dataset/old_wav_ful' を再帰的に削除しました。
フォルダ './../dataset/old_txt_ful' を再帰的に削除しました。
